# Filter Tren Per-Coin (ulang, dengan redistribusi) — Terbukti Tidak Membantu V2-60

**Latar belakang**: spesifikasi "Sistem Manajemen Aset Crypto" (30 Sep 2026) minta filter tren per-coin
*baru* ditambahkan ke V2-60 — coin dengan tren jelek tidak dapat alokasi bulan itu, porsinya (maks 10%)
dipindah ke coin lain yang lolos filter.

**Masalah**: notebook 15 *sudah* mencoba ide serupa (varian V3: coin `close < SMA100` sendiri → porsi
jadi **cash**) dan **menolaknya** — Sharpe uji turun dari 1.16 jadi 0.77–0.88, disimpulkan "rawan overfit".

Notebook ini menguji ulang dengan 2 perbedaan dari spek baru yang belum pernah dites:
1. **Indikator dicari secara empiris** (bukan SMA100 yang diasumsikan) lewat proxy forward-return per coin.
2. **Mekanisme redistribusi** (porsi coin gagal → pindah ke coin lain yang lolos, inverse-vol), bukan cash-out.

Plus 4 variasi desain tambahan: trigger harian vs dikunci bulanan, filter persisten (gagal 2 bulan
beruntun), kombinasi mayoritas 3 indikator, dan filter *scaled* (bobot berkurang bertahap, bukan on/off).

**Metodologi** (ikut aturan notebook 15, lihat README bagian 4): indikator & ambang dipilih **hanya**
dari periode dev **2018-07 → 2022-12**; periode uji **2023-01 → 2026-06** dijalankan **sekali** per
desain terpilih, tidak diubah-ubah setelah dilihat.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from data_source.loader import load_symbol
from data_source.fear_greed import load_fear_greed_index
from research.single_position import Panel, BTC
from research.portfolio import run_portfolio, inverse_vol_weights, apply_caps

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

UNIV = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ADAUSDT", "LINKUSDT", "TRXUSDT", "DOGEUSDT"]
CAPS = {"DOGEUSDT": 0.10}
DEV, TEST = ("2018-07-01", "2022-12-31"), ("2023-01-01", "2026-06-30")

P = Panel({s: load_symbol(s, "1d", "since2018") for s in UNIV}, "2018-01-01", "2026-06-30")
fgi = load_fear_greed_index()["fng_value"].reindex(P.dates).ffill()
btc = P.close[BTC]
market = (btc > btc.rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50)   # filter H8b, tidak berubah
base_w = inverse_vol_weights(P, UNIV, 60, CAPS)
W_V2_60 = base_w.mul(market.astype(float), axis=0) * 0.6        # baseline: V2-60 tanpa filter per-coin
print("Setup selesai.", P.dates[0].date(), "->", P.dates[-1].date())

Setup selesai. 2018-01-01 -> 2026-06-30


## 1. Cari indikator per-coin (proxy: forward-return bulanan, DEV saja)

Bukan mengasumsikan satu indikator — hitung banyak kandidat (rasio ke SMA sendiri, momentum,
relative-strength ke BTC, jarak dari puncak N hari, RSI), lalu lihat mana yang **secara empiris**
berkorelasi dengan return bulan berikutnya, **hanya di periode dev** (2018-07 → 2022-12).

In [2]:
close = P.close
dates_dev = P.dates[(P.dates >= DEV[0]) & (P.dates <= DEV[1])]
month_starts = dates_dev[~dates_dev.to_period("M").duplicated()]
month_starts = month_starts[month_starts < dates_dev[-1]]

sma_ratio = {n: close / P.ma(n) - 1 for n in (20, 50, 100, 150, 200)}
momentum = {n: P.ret(n) for n in (30, 60, 90)}
drawdown_from_high = {n: close / close.rolling(n, min_periods=n).max() - 1 for n in (30, 60, 90)}
rsi14 = P.rsi(14)
btc_mom = {n: momentum[n][BTC] for n in (30, 60, 90)}
rel_strength = {n: momentum[n].sub(btc_mom[n], axis=0) for n in (30, 60, 90)}

candidates = {}
for n in (20, 50, 100, 150, 200):
    candidates[f"sma_ratio_{n}"] = sma_ratio[n]
for n in (30, 60, 90):
    candidates[f"momentum_{n}"] = momentum[n]
    candidates[f"drawdown_from_high_{n}"] = drawdown_from_high[n]
    candidates[f"rel_strength_vs_btc_{n}"] = rel_strength[n]
candidates["rsi14"] = rsi14

rows = []
for i in range(len(month_starts) - 1):
    r, r_next = month_starts[i], month_starts[i + 1]
    for s in UNIV:
        c0, c1 = close.at[r, s], close.at[r_next, s]
        if pd.isna(c0) or pd.isna(c1):
            continue
        row = {"coin": s, "date": r, "fwd_ret": c1 / c0 - 1}
        for name, df in candidates.items():
            row[name] = df.at[r, s] if s in df.columns else np.nan
        rows.append(row)
data_dev_proxy = pd.DataFrame(rows)
print(f"Sampel dev: {len(data_dev_proxy)} baris (coin x bulan), {data_dev_proxy['date'].min().date()} .. {data_dev_proxy['date'].max().date()}")

results = []
for name in candidates:
    d = data_dev_proxy[["fwd_ret", name]].dropna()
    if len(d) < 30:
        continue
    corr, pval = spearmanr(d[name], d["fwd_ret"])
    med = d[name].median()
    hi, lo = d[d[name] > med], d[d[name] <= med]
    results.append(dict(indikator=name, n=len(d), spearman=corr, pval=pval,
                         ret_atas_median=hi["fwd_ret"].mean(), ret_bawah_median=lo["fwd_ret"].mean(),
                         spread_median=hi["fwd_ret"].mean() - lo["fwd_ret"].mean()))
res_ind = pd.DataFrame(results).sort_values("spearman", ascending=False)
out = res_ind.copy().astype(object)
for c in ("spearman",):
    out[c] = res_ind[c].map(lambda v: f"{v:+.3f}")
out["pval"] = res_ind["pval"].map(lambda v: f"{v:.3f}")
for c in ("ret_atas_median", "ret_bawah_median", "spread_median"):
    out[c] = res_ind[c].map(lambda v: f"{v:+.2%}")
out

Sampel dev: 404 baris (coin x bulan), 2018-07-01 .. 2022-11-01


,indikator,n,spearman,pval,ret_atas_median,ret_bawah_median,spread_median
0,sma_ratio_20,403,+0.151,0.002,+16.84%,+3.73%,+13.11%
14,rsi14,404,+0.118,0.018,+15.65%,+4.85%,+10.80%
6,drawdown_from_high_30,401,+0.093,0.063,+17.76%,+3.05%,+14.70%
5,momentum_30,401,+0.087,0.083,+17.93%,+2.89%,+15.04%
1,sma_ratio_50,400,+0.076,0.131,+17.04%,+3.64%,+13.40%
11,momentum_90,392,+0.065,0.199,+17.06%,+4.40%,+12.66%
4,sma_ratio_200,372,+0.046,0.380,+18.58%,+2.70%,+15.89%
9,drawdown_from_high_60,397,+0.046,0.365,+12.57%,+8.41%,+4.16%
2,sma_ratio_100,392,+0.028,0.587,+16.51%,+4.96%,+11.56%
3,sma_ratio_150,384,+0.024,0.637,+15.13%,+4.91%,+10.22%


**Hasil**: indikator jangka panjang (SMA100/150/200, relative-strength ke BTC) lemah atau nol —
tidak memisahkan bulan bagus vs jelek. Yang menonjol (signifikan, konsisten tanda positif):
**jarak dari puncak harga 30/60 hari** (`drawdown_from_high_30/60`) dan **harga vs SMA-20 sendiri**.
Dua ini yang akan diuji di engine portofolio penuh (bukan cuma proxy coin-level di atas — efek
redistribusi, fee, dan interaksi dengan filter market cuma kelihatan di simulasi portofolio asli).

In [3]:
def inverse_vol_weights_filtered(panel, syms, lookback, caps, eligible):
    '''Sama seperti inverse_vol_weights, tapi coin yg eligible[s]==False di hari itu -> bobot 0,
    porsinya otomatis terdistribusi ke coin lain yg eligible (lewat renormalisasi) -- BUKAN cash-out
    seperti notebook 15 V3.'''
    vol = panel.close[syms].pct_change().rolling(lookback, min_periods=lookback).std()
    raw = (1 / vol).replace([np.inf, -np.inf], np.nan)
    raw = raw.where(eligible[syms], 0.0)
    w = raw.div(raw.sum(axis=1), axis=0).fillna(0.0)
    return apply_caps(w, caps)


def weights_from_scale(scale):
    '''scale: DataFrame 0..1 per coin per hari (0=sangat tidak sehat, 1=sehat penuh) -- filter bertahap.'''
    vol = P.close[UNIV].pct_change().rolling(60, min_periods=60).std()
    raw = (1 / vol).replace([np.inf, -np.inf], np.nan) * scale[UNIV].clip(0, 1)
    w = raw.div(raw.sum(axis=1), axis=0).fillna(0.0)
    return apply_caps(w, CAPS)


def monthly_lock(x):
    '''Kunci nilai x di tanggal 1 tiap bulan, berlaku sepanjang bulan -- filter tidak dicek ulang
    harian, supaya tidak nambah rebalance di luar jadwal bulanan.'''
    return x.resample("MS").first().reindex(x.index, method="ffill")


def stats(eq, info, label):
    years = (eq.index[-1] - eq.index[0]).days / 365.25
    tot = eq.iloc[-1] / eq.iloc[0] - 1
    dd = (eq / eq.cummax() - 1).min()
    dr = eq.pct_change().dropna()
    mon = eq.resample("ME").last().pct_change().dropna()
    return dict(strategi=label, profit=tot, cagr=(1 + tot) ** (1 / years) - 1, max_dd=dd,
                sharpe=dr.mean() / dr.std() * np.sqrt(365) if dr.std() > 0 else np.nan,
                bulan_untung=(mon > 0).mean(), bulan_terburuk=mon.min(), n_rebalance=info["n_rebalance"])


def show(res):
    out = res.copy().astype(object)
    for c in ("profit", "cagr", "max_dd", "bulan_untung", "bulan_terburuk"):
        out[c] = res[c].map(lambda v: f"{v:+.0%}")
    out["sharpe"] = res["sharpe"].map(lambda v: f"{v:.2f}")
    return out


dd30 = P.close / P.close.rolling(30, min_periods=30).max() - 1
dd60 = P.close / P.close.rolling(60, min_periods=60).max() - 1
sma20r = P.close / P.ma(20) - 1
print("Helper siap.")

Helper siap.


## 2. Uji di engine portofolio penuh — trigger harian (sesuai bunyi literal spek)

Spek bilang "rebalance ... plus setiap kali status filter ... per-coin berubah" — diuji literal dulu:
filter dicek **tiap hari**, rebalance terjadi tiap kali status eligible berubah (persis mekanisme
`run_portfolio` yang sudah ada buat filter market).

In [4]:
candidates_daily = {}
for th in (-0.10, -0.15, -0.20, -0.25, -0.30):
    candidates_daily[f"dd30 > {th:+.0%}"] = dd30 > th
for th in (-0.10, -0.15, -0.20, -0.25, -0.30):
    candidates_daily[f"dd60 > {th:+.0%}"] = dd60 > th
for th in (-0.10, -0.05, 0.0):
    candidates_daily[f"sma20ratio > {th:+.0%}"] = sma20r > th

rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 (baseline, tanpa filter per-coin)")]
for name, elig in candidates_daily.items():
    W = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, elig).mul(market.astype(float), axis=0) * 0.6
    rows.append(stats(*run_portfolio(P, W, *DEV), f"+filter {name}"))
dev_daily = pd.DataFrame(rows).set_index("strategi")
show(dev_daily)

,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
"V2-60 (baseline, tanpa filter per-coin)",+1478%,+85%,-29%,1.65,+40%,-15%,96
+filter dd30 > -10%,+925%,+68%,-27%,1.66,+40%,-11%,431
+filter dd30 > -15%,+774%,+62%,-33%,1.58,+40%,-18%,388
+filter dd30 > -20%,+685%,+58%,-34%,1.51,+40%,-23%,290
+filter dd30 > -25%,+842%,+65%,-34%,1.60,+40%,-20%,227
+filter dd30 > -30%,+854%,+65%,-34%,1.57,+40%,-18%,200
+filter dd60 > -10%,+1093%,+73%,-25%,1.77,+40%,-12%,422
+filter dd60 > -15%,+823%,+64%,-34%,1.62,+40%,-18%,393
+filter dd60 > -20%,+698%,+59%,-36%,1.52,+40%,-20%,335


In [5]:
best_daily = dev_daily.drop(index="V2-60 (baseline, tanpa filter per-coin)")["sharpe"].idxmax()
print(f"Terpilih dari dev (Sharpe tertinggi): {best_daily}")
chosen_elig = candidates_daily[best_daily.replace("+filter ", "")]
W_chosen = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, chosen_elig).mul(market.astype(float), axis=0) * 0.6
rows_test = [stats(*run_portfolio(P, W_V2_60, *TEST), "V2-60 (baseline, tanpa filter per-coin)"),
             stats(*run_portfolio(P, W_chosen, *TEST), best_daily)]
test_daily = pd.DataFrame(rows_test).set_index("strategi")
show(test_daily)

Terpilih dari dev (Sharpe tertinggi): +filter dd60 > -10%


,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
"V2-60 (baseline, tanpa filter per-coin)",+157%,+31%,-21%,1.17,+44%,-11%,74
+filter dd60 > -10%,+78%,+18%,-28%,0.87,+44%,-9%,408


**Hasil trigger harian**: jumlah rebalance melonjak (~5-6x lipat dari ~74 jadi ~400+) karena status
filter per-coin sering berubah harian. Di periode uji, hasilnya **lebih buruk** dari baseline di semua
metrik (profit, Sharpe, bahkan max drawdown lebih dalam) — ongkos transaksi tambahan dari keluar-masuk
lebih sering memakan keuntungan filter, persis kekhawatiran di dokumen spek. Lanjut coba versi yang
tidak menambah turnover.

## 3. Uji dikunci bulanan (filter dicek cuma pas tanggal 1, tidak nambah rebalance)

In [6]:
candidates_m = {}
for th in (-0.10, -0.15, -0.20, -0.25, -0.30):
    candidates_m[f"dd30 > {th:+.0%} (bulanan)"] = monthly_lock(dd30 > th)
for th in (-0.10, -0.15, -0.20, -0.25, -0.30):
    candidates_m[f"dd60 > {th:+.0%} (bulanan)"] = monthly_lock(dd60 > th)

rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 (baseline)")]
for name, elig in candidates_m.items():
    W = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, elig).mul(market.astype(float), axis=0) * 0.6
    rows.append(stats(*run_portfolio(P, W, *DEV), f"+filter {name}"))
dev_m = pd.DataFrame(rows).set_index("strategi")
show(dev_m)

,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
V2-60 (baseline),+1478%,+85%,-29%,1.65,+40%,-15%,96
+filter dd30 > -10% (bulanan),+1492%,+85%,-34%,1.67,+36%,-23%,110
+filter dd30 > -15% (bulanan),+1314%,+80%,-33%,1.63,+38%,-14%,111
+filter dd30 > -20% (bulanan),+1243%,+78%,-31%,1.62,+38%,-15%,118
+filter dd30 > -25% (bulanan),+1334%,+81%,-30%,1.64,+38%,-15%,111
+filter dd30 > -30% (bulanan),+1331%,+81%,-29%,1.66,+40%,-15%,108
+filter dd60 > -10% (bulanan),+1202%,+77%,-33%,1.57,+34%,-23%,102
+filter dd60 > -15% (bulanan),+1107%,+74%,-27%,1.58,+36%,-14%,108
+filter dd60 > -20% (bulanan),+1264%,+79%,-26%,1.66,+36%,-14%,108


In [7]:
best_m = dev_m.drop(index="V2-60 (baseline)")["sharpe"].idxmax()
print(f"Terpilih dari dev (Sharpe tertinggi): {best_m}")
chosen_elig_m = candidates_m[best_m.replace("+filter ", "")]
W_chosen_m = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, chosen_elig_m).mul(market.astype(float), axis=0) * 0.6
rows_test = [stats(*run_portfolio(P, W_V2_60, *TEST), "V2-60 (baseline)"),
             stats(*run_portfolio(P, W_chosen_m, *TEST), best_m)]
test_m = pd.DataFrame(rows_test).set_index("strategi")
show(test_m)

Terpilih dari dev (Sharpe tertinggi): +filter dd60 > -30% (bulanan)


,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
V2-60 (baseline),+157%,+31%,-21%,1.17,+44%,-11%,74
+filter dd60 > -30% (bulanan),+153%,+30%,-23%,1.16,+44%,-11%,83


**Hasil dikunci bulanan**: turnover kembali normal, tapi hasil uji **nyaris identik** dengan baseline
(selisih profit/Sharpe <1 poin persen, drawdown malah sedikit lebih dalam). "Pemenang" dari dev
(`dd60 > -30%`) ambangnya longgar sekali — nyaris tidak pernah menyaring coin manapun — tanda bahwa
keunggulan tipis di dev itu kemungkinan noise dari banyak kombinasi yang dicoba, bukan sinyal asli.

## 4. Variasi lanjutan: persisten, kombinasi mayoritas, dan scaled

Tiga desain tambahan yang mungkin mengurangi *whipsaw* dibanding filter on/off snapshot bulanan biasa.

In [8]:
# 4a. Persisten -- gagal 2 bulan beruntun baru dikeluarkan (grace period 1 bulan)
cand_persist = {}
for name, raw_ind in (("dd30", dd30), ("dd60", dd60), ("sma20r", sma20r)):
    for th in (-0.10, -0.15, -0.20, -0.25):
        m = monthly_lock(raw_ind > th)
        fail_now = ~m
        fail_prev = fail_now.shift(1).fillna(False)
        disable = fail_now & fail_prev
        cand_persist[f"{name}>{th:+.0%} persisten2bln"] = ~disable

rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 (baseline)")]
for name, elig in cand_persist.items():
    W = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, elig).mul(market.astype(float), axis=0) * 0.6
    rows.append(stats(*run_portfolio(P, W, *DEV), name))
dev_persist = pd.DataFrame(rows).set_index("strategi")
best_persist = dev_persist.drop(index="V2-60 (baseline)")["sharpe"].idxmax()
print(f"Terpilih dev (persisten): {best_persist}")
show(dev_persist)

Terpilih dev (persisten): dd60>-25% persisten2bln


,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
V2-60 (baseline),+1478%,+85%,-29%,1.65,+40%,-15%,96
dd30>-10% persisten2bln,+1661%,+89%,-34%,1.71,+42%,-25%,117
dd30>-15% persisten2bln,+1255%,+78%,-32%,1.68,+40%,-14%,118
dd30>-20% persisten2bln,+1123%,+74%,-31%,1.65,+38%,-15%,121
dd30>-25% persisten2bln,+1214%,+77%,-30%,1.67,+38%,-15%,111
dd60>-10% persisten2bln,+1328%,+81%,-34%,1.61,+40%,-25%,108
dd60>-15% persisten2bln,+1024%,+71%,-26%,1.60,+38%,-14%,116
dd60>-20% persisten2bln,+1152%,+75%,-26%,1.69,+36%,-14%,113
dd60>-25% persisten2bln,+1318%,+80%,-29%,1.72,+38%,-14%,113


In [9]:
# 4b. Kombinasi mayoritas -- >=2 dari 3 indikator harus bilang sehat
cand_vote = {}
for th_dd in (-0.15, -0.20):
    for th_sma in (-0.05, 0.0):
        votes = (monthly_lock(dd30 > th_dd).astype(int) + monthly_lock(dd60 > th_dd).astype(int)
                 + monthly_lock(sma20r > th_sma).astype(int))
        cand_vote[f"dd>{th_dd:+.0%} & sma>{th_sma:+.0%}, >=2/3"] = votes >= 2
        cand_vote[f"dd>{th_dd:+.0%} & sma>{th_sma:+.0%}, 3/3"] = votes >= 3

rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 (baseline)")]
for name, elig in cand_vote.items():
    W = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, elig).mul(market.astype(float), axis=0) * 0.6
    rows.append(stats(*run_portfolio(P, W, *DEV), name))
dev_vote = pd.DataFrame(rows).set_index("strategi")
best_vote = dev_vote.drop(index="V2-60 (baseline)")["sharpe"].idxmax()
print(f"Terpilih dev (mayoritas): {best_vote}")
show(dev_vote)

Terpilih dev (mayoritas): dd>-20% & sma>+0%, >=2/3


,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
V2-60 (baseline),+1478%,+85%,-29%,1.65,+40%,-15%,96
"dd>-15% & sma>-5%, >=2/3",+1338%,+81%,-33%,1.64,+38%,-14%,111
"dd>-15% & sma>-5%, 3/3",+1102%,+74%,-30%,1.56,+36%,-17%,108
"dd>-15% & sma>+0%, >=2/3",+1295%,+80%,-34%,1.63,+38%,-14%,112
"dd>-15% & sma>+0%, 3/3",+801%,+63%,-30%,1.40,+32%,-17%,106
"dd>-20% & sma>-5%, >=2/3",+1353%,+81%,-31%,1.67,+38%,-15%,115
"dd>-20% & sma>-5%, 3/3",+1190%,+77%,-26%,1.63,+36%,-14%,106
"dd>-20% & sma>+0%, >=2/3",+1493%,+85%,-30%,1.71,+38%,-15%,111
"dd>-20% & sma>+0%, 3/3",+876%,+66%,-27%,1.45,+34%,-14%,106


In [10]:
# 4c. Scaled -- bobot berkurang bertahap (ramp linear), bukan on/off
cand_scale = {}
for lo, hi in ((-0.30, -0.05), (-0.25, -0.10), (-0.20, -0.05), (-0.30, -0.15)):
    scale = ((dd30 - lo) / (hi - lo)).clip(0, 1)
    cand_scale[f"dd30 ramp {lo:+.0%}->{hi:+.0%}"] = monthly_lock(scale)

rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 (baseline)")]
for name, sc in cand_scale.items():
    W = weights_from_scale(sc).mul(market.astype(float), axis=0) * 0.6
    rows.append(stats(*run_portfolio(P, W, *DEV), name))
dev_scale = pd.DataFrame(rows).set_index("strategi")
best_scale = dev_scale.drop(index="V2-60 (baseline)")["sharpe"].idxmax()
print(f"Terpilih dev (scaled): {best_scale}")
show(dev_scale)

Terpilih dev (scaled): dd30 ramp -30%->-15%


,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
V2-60 (baseline),+1478%,+85%,-29%,1.65,+40%,-15%,96
dd30 ramp -30%->-5%,+1463%,+84%,-28%,1.69,+38%,-14%,108
dd30 ramp -25%->-10%,+1356%,+81%,-32%,1.64,+36%,-15%,111
dd30 ramp -20%->-5%,+1380%,+82%,-34%,1.65,+38%,-15%,118
dd30 ramp -30%->-15%,+1444%,+84%,-27%,1.69,+38%,-15%,108


### Uji sekali di periode uji: ketiga pemenang dev vs baseline

In [11]:
W_persist = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, cand_persist[best_persist]).mul(market.astype(float), axis=0) * 0.6
W_vote = inverse_vol_weights_filtered(P, UNIV, 60, CAPS, cand_vote[best_vote]).mul(market.astype(float), axis=0) * 0.6
W_scale = weights_from_scale(cand_scale[best_scale]).mul(market.astype(float), axis=0) * 0.6

rows_test = [
    stats(*run_portfolio(P, W_V2_60, *TEST), "V2-60 (baseline, tanpa filter per-coin)"),
    stats(*run_portfolio(P, W_persist, *TEST), f"Persisten: {best_persist}"),
    stats(*run_portfolio(P, W_vote, *TEST), f"Mayoritas: {best_vote}"),
    stats(*run_portfolio(P, W_scale, *TEST), f"Scaled: {best_scale}"),
]
test_final = pd.DataFrame(rows_test).set_index("strategi")
show(test_final)

,profit,cagr,max_dd,sharpe,bulan_untung,bulan_terburuk,n_rebalance
strategi,,,,,,,
"V2-60 (baseline, tanpa filter per-coin)",+157%,+31%,-21%,1.17,+44%,-11%,74
Persisten: dd60>-25% persisten2bln,+151%,+30%,-24%,1.17,+44%,-11%,91
"Mayoritas: dd>-20% & sma>+0%, >=2/3",+137%,+28%,-26%,1.11,+41%,-11%,92
Scaled: dd30 ramp -30%->-15%,+154%,+31%,-23%,1.17,+44%,-11%,80


## Kesimpulan

**Filter tren per-coin, dalam 5 desain berbeda yang dites (snapshot harian, dikunci bulanan, persisten
2 bulan, kombinasi mayoritas 3 indikator, dan scaled/bertahap), tidak satupun terbukti memperbaiki
V2-60 di periode uji 2023-01 → 2026-06** — sejalan dan memperkuat kesimpulan notebook 15, meski kali
ini dengan indikator yang sudah terbukti lebih baik (`drawdown_from_high_30/60`, bukan SMA100 yang
diasumsikan) dan mekanisme redistribusi (bukan cash-out).

Pola yang konsisten di semua varian:
1. **Trigger harian merusak lewat turnover** — rebalance melonjak 5-6x, fee tambahan memakan hasil
   lebih dari manfaat filter (profit/Sharpe/drawdown semua lebih buruk dari baseline).
2. **Dikunci bulanan menghilangkan masalah turnover, tapi hasilnya jadi nyaris identik dengan baseline**
   (dalam radius noise) — filter tidak menambah apa pun yang berarti.
3. **Persisten, mayoritas, dan scaled** (desain yang didesain khusus mengurangi *whipsaw*) tetap tidak
   mengalahkan baseline secara meyakinkan di periode uji — lihat tabel di atas.
4. Dari eksplorasi tahun-per-tahun (lihat riset awal, sebelum notebook ini), filter berbasis harga juga
   **tidak membantu sama sekali saat crash sistemik 2022** (LUNA/FTX) — ke-8 coin jatuh bersamaan,
   proteksi crash tetap jadi tugas filter market H8b yang sudah ada, bukan filter per-coin.

### Rekomendasi
**Jangan tambahkan filter tren per-coin ke V2-60.** Diversifikasi 8-coin + inverse-vol + filter market
H8b yang sudah ada sudah menangani perannya, dan setiap variasi filter per-coin yang dicoba (di notebook
ini maupun notebook 15) menambah kompleksitas & risiko overfit tanpa bukti hasil lebih baik. Fokus
pengembangan selanjutnya untuk spek baru sebaiknya ke **trailing-stop per-coin** (bagian spek yang
belum pernah diuji sama sekali), bukan filter tren bulanan.

### Catatan metodologi
- Indikator (bagian 1) dan ambang (bagian 2-4) dipilih **hanya** dari periode dev 2018-07→2022-12;
  periode uji dijalankan sekali per desain terpilih dan tidak diubah usai dilihat.
- ~30+ kombinasi (indikator × ambang × mekanisme) dicoba di notebook ini — makin banyak dicoba, makin
  besar peluang satu kombinasi "menang" di dev secara kebetulan (lihat README bagian 4 poin 3). Fakta
  bahwa **semua desain gagal mengalahkan baseline secara meyakinkan di uji**, meski beberapa sempat
  unggul tipis di dev, adalah bukti tambahan bahwa keunggulan di dev itu kemungkinan besar noise.